In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import linregress, gaussian_kde, mode
from scipy.optimize import brentq, curve_fit

from statsmodels.stats.proportion import proportion_confint

import powerlaw

import os
import glob

import uncvalue

import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.patches import Patch

# warnings
import warnings
warnings.filterwarnings("ignore")

def C_random(sigma, d, S):
    return (d/sigma)**2 / S

def C_mutualistic(sigma, d, S):
    return d / (sigma * (S - 1) * np.sqrt(2.0 / np.pi))

def competitive_lhs(C, sigma, S):
    term1 = np.sqrt(S * C)
    term2 = 1.0 + (2.0 - 2.0 * C) / (np.pi - 2.0 * C)
    term3 = np.sqrt((np.pi - 2.0 * C) / np.pi)
    term4 = C * np.sqrt(2.0 / np.pi)
    return sigma * (term1 * term2 * term3 + term4)


def C_competitive(sigma, d, S, C_bounds=(1e-12, 0.999)):
    try:
        f = lambda C: competitive_lhs(C, sigma, S) - d
        return brentq(f, *C_bounds)
    except ValueError:
        return np.nan

def C_predator_prey(sigma, d, S):
    return (d / (sigma * (np.pi - 2.0) / np.pi)) ** 2 / S

def C_mixture(sigma, d, S):
    return (d / (sigma * (np.pi + 2.0) / np.pi)) ** 2 / S

def C_spatial(sigma, d, S, n, ρ):
    return (d/sigma)**2 * (n / (1+ (n-1)*ρ)) * (1 / (S-1)) 

def C_feasibility(sigma, d, S):

    return 0.5*(d/sigma)**2 / S

def power_law(x, a, b):
    return a * np.power(x, b)


# Color code for each network type (foodwebs, plant-pollinator...)
color_dict = {
    "food_web": "C0",
    "anemone_fish": "C1",
    "plant_pollinator": "C2",
    "host_parasite": "C3",
    "seed_dispersal": "C4",
    "plant_ant": "C5",
}

net_type_initials = {
    "food_web": "FW",
    "anemone_fish": "AF",
    "plant_pollinator": "PP",
    "host_parasite": "HP",
    "seed_dispersal": "SD",
    "plant_ant": "PA",
}

# $C\sim S^{-1}$ scaling

Here we show that all results derived from RMT (or similar) give rise to $C\sim S^{-1}$ scaling, and that this scaling is observed in nature

In [ ]:
filenames = glob.glob("Results/efficiency/perturbations/1%/*.csv")

df_perturbations = pd.DataFrame()

for filename in filenames:
    df_i = pd.read_csv(filename)

    df_perturbations = pd.concat([df_perturbations, df_i], ignore_index=True)

df_plot = pd.read_csv("Results/network_metrics.csv")

df_plot.dropna(subset=["C"], inplace=True)

# Remove network_type="other"
df_plot = df_plot[df_plot["network_type"] != "other"]

SC = np.mean(df_plot["S"] * df_plot["C"])

df_natcomm = pd.read_csv("Data/NatComm_data.csv")

In [ ]:
fig, ax = plt.subplot_mosaic(
    """
    AB
    CD
    """,
    figsize=(8 * 2, 6 * 2),
    gridspec_kw={"hspace": 0.3, "wspace": 0.3},
)

letters = ["A", "B", "C", "D"]

# Complexity-stability limit with theory
sigma = 1
d = 1.0

Ss = np.logspace(0, 5, 100)

Cs_random = []
Cs_mutualistic = []
Cs_competitive = []
Cs_predator_prey = []
Cs_mixture = []
Cs_feasibility = []
Cs_spatial = []

for S in Ss:

    Cs_random.append(C_random(sigma, d, S))
    Cs_mutualistic.append(C_mutualistic(sigma, d, S))
    Cs_competitive.append(C_competitive(sigma, d, S))
    Cs_predator_prey.append(C_predator_prey(sigma, d, S))
    Cs_mixture.append(C_mixture(sigma, d, S))
    Cs_feasibility.append(C_feasibility(sigma, d, S))
    Cs_spatial.append(C_spatial(sigma, d, S, n=10, ρ=0.5))

ax["A"].plot(Ss, Cs_random, lw=5, color="purple", label="Random")
ax["A"].plot(Ss, Cs_mutualistic, lw=5, color="mediumseagreen", label="Mutualistic")
ax["A"].plot(Ss, Cs_competitive, lw=5, color="salmon", label="Competitive")
ax["A"].plot(Ss, Cs_predator_prey, lw=5, color="steelblue", label="Predator-prey")
ax["A"].plot(Ss, Cs_mixture, lw=5, color="teal", label="Mixture")
ax["A"].plot(Ss, Cs_feasibility, lw=5, color="orange", label="Feasibility")
ax["A"].plot(Ss, Cs_spatial, lw=5, color="darkgreen", label="Spatial")
x = np.logspace(0, 5, 100)
y = 2 * np.power(x, -1)

ax["A"].plot(x, y, lw=3, color="k", ls="--", label=r"$C \sim S^{-1}$")

ax["A"].set_xscale("log")
ax["A"].set_yscale("log")

ax["A"].set_xlabel(r"Number of species ($S$)", fontsize=24)
ax["A"].set_ylabel(r"Connectance ($C$)", fontsize=24)

ax["A"].legend(fontsize=14, ncol=2)

# Complexity-stability limit with data
tot_S = df_plot["S"].values.astype(float)
tot_C = df_plot["C"].values.astype(float)

reg = linregress(np.log(tot_S), np.log(tot_C))

for n_type in df_plot["network_type"].unique():

    subset = df_plot[df_plot["network_type"] == n_type]

    ax["B"].scatter(
        subset["S"],
        subset["C"],
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
    )

# ax["B"].scatter(df_natcomm["S"], df_natcomm["C"], label="NatComm", color="pink", marker="o", s=12)

lr = linregress(np.log(tot_S), np.log(tot_C))

print("Slope: {:.2f}, Intercept: {:.2f}, R^2: {:.2f}, p-value: {:.2e}".format(lr.slope, lr.intercept, lr.rvalue**2, lr.pvalue))

# Print slope +- 95% CI
print("Slope: {:.2f} [{:.2f}, {:.2f}]".format(lr.slope, lr.slope - 1.96 * lr.stderr, lr.slope + 1.96 * lr.stderr))

x = np.logspace(0, 3, 100)
y = SC * np.power(x, -1)

ax["B"].plot(x, y, lw=3, color="k", label=r"$C \sim S^{-1}$")

ax["B"].plot(x, np.exp(reg.intercept) * np.power(x, reg.slope), lw=3, color="k", ls="--", label=r"Data fit: $C \sim S^{%.2f}$" % reg.slope)

ax["B"].set_xscale("log")
ax["B"].set_yscale("log")

ax["B"].set_xlabel(r"Number of species ($S$)", fontsize=24)
ax["B"].set_ylabel(r"Connectance ($C$)", fontsize=24)

ax["B"].legend(fontsize=14)

# Mean degree vs species richness
for n_type in df_plot["network_type"].unique():

    subset = df_plot[df_plot["network_type"] == n_type]

    ax["C"].scatter(
        subset["S"],
        subset["mean_degree"],
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
    )

x = df_plot["S"].values.astype(float)
y = df_plot["mean_degree"].values.astype(float)

res = linregress(np.log(x), np.log(y))

r_squared = res.rvalue**2
p_value = res.pvalue

p_label = r"$p < 0.001$" if p_value < 0.001 else "p = {:.2e}".format(p_value)

x = np.logspace(0, 3, 100)

ax["C"].plot(
    x,
    np.exp(res.intercept) * np.power(x, res.slope),
    lw=3,
    color="k",
    ls="--",
    label=r"$\langle k \rangle \sim S^{%.2f}$" % res.slope,
)

ax["C"].text(
    0.05,
    0.8,
    r"$R^2 = {:.2f}$".format(r_squared),
    transform=ax["C"].transAxes,
    fontsize=14,
)
ax["C"].text(
    0.05,
    0.7,
    p_label,
    transform=ax["C"].transAxes,
    fontsize=14,
)


ax["C"].set_xscale("log")
ax["C"].set_yscale("log")

ax["C"].set_xlabel(r"Number of species ($S$)", fontsize=24)
ax["C"].set_ylabel(r"Mean degree ($\langle k \rangle$)", fontsize=24)
ax["C"].legend(fontsize=14)

# L vs S
# Use the linear regression instead of curve_fit for better confidence intervals
lr = linregress(
    np.log(df_plot["S"].values.astype(float)), np.log(df_plot["L"].values.astype(float))
)

print(lr)

# Use uncvalue for uncertainty propagation
intercept = uncvalue.Value(lr.intercept, np.sqrt(lr.intercept_stderr**2))
slope = uncvalue.Value(lr.slope, np.sqrt(lr.stderr**2))

c = np.exp(intercept)
b = slope

c_min_95_CI = c - 1.96 * c.unc
c_max_95_CI = c + 1.96 * c.unc

slope_min_95_CI = slope - 1.96 * slope.unc
slope_max_95_CI = slope + 1.96 * slope.unc

slope = slope.val
slope_min_95_CI = slope_min_95_CI.val
slope_max_95_CI = slope_max_95_CI.val

df_compute = df_plot[df_plot["S"] > 5]

δ_opt = np.mean(
    np.log(3 / (df_compute["diffusion_time"] * c.val)) / np.log(df_compute["S"])
)
delta_opt_min_95_CI = np.mean(
    np.log(3 / (df_compute["diffusion_time"] * c_min_95_CI)) / np.log(df_compute["S"])
)
delta_opt_max_95_CI = np.mean(
    np.log(3 / (df_compute["diffusion_time"] * c_max_95_CI)) / np.log(df_compute["S"])
)

expected_exponent = 1 + δ_opt
expected_exponent_min_95_CI = 1 + delta_opt_min_95_CI
expected_exponent_max_95_CI = 1 + delta_opt_max_95_CI

for n_type in df_plot["network_type"].unique():
    
    subset = df_plot[df_plot["network_type"] == n_type]

    ax["D"].scatter(
        subset["S"],
        subset["L"],
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
    )


x = np.logspace(0, 3.5, 100)
y = c * np.power(x, slope)

p_label = r"$p < 0.001$" if lr.pvalue < 0.001 else "p = {:.2e}".format(lr.pvalue)

ax["D"].plot(x, y, lw=3, color="k", label=r"$L=cS^{\gamma}$")

ax["D"].text(
    0.05,
    0.8,
    r"$\gamma$={:.2f} [{:.2f}, {:.2f}] ($R^2$={}, {})".format(
        slope,
        slope_min_95_CI,
        slope_max_95_CI,
        np.round(lr.rvalue**2, 2),
        p_label,
    ),
    transform=ax["D"].transAxes,
    fontsize=14,
)
# ax["D"].text(
#     0.05,
#     0.7,
#     r"$\gamma_{{opt}}$: {:.2f} [{:.2f}, {:.2f}]".format(
#         expected_exponent, expected_exponent_min_95_CI, expected_exponent_max_95_CI
#     ),
#     transform=ax["D"].transAxes,
#     fontsize=14,
# )

ax["D"].set_xscale("log")
ax["D"].set_yscale("log")

ax["D"].set_xlabel(r"Number of species ($S$)", fontsize=24)
ax["D"].set_ylabel(r"Number of interactions ($L$)", fontsize=24)

ax["D"].legend(fontsize=14)

# Custom legend for network types
handles = []
labels = []
for n_type, color in color_dict.items():
    handles.append(plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=10))
    labels.append(n_type.replace("_", " ").title())

fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=14)

for letter in letters:
    ax[letter].tick_params(axis="both", which="major", labelsize=16)
    ax[letter].tick_params(axis="both", which="minor", labelsize=16)

    # Add subplot labels
    if (letter == "C") or (letter == "D"):
        ax[letter].text(
            0.85,
            0.15,
            "({})".format(letter.lower()),
            transform=ax[letter].transAxes,
            fontsize=24,
            fontweight="bold",
            va="top",
        )

    else:
        ax[letter].text(
            0.05,
            0.15,
            "({})".format(letter.lower()),
            transform=ax[letter].transAxes,
            fontsize=24,
            fontweight="bold",
            va="top",
        )

# Network efficiency

In [ ]:
results_efficiency = {}
results_efficiency_ER = {}
results_efficiency_CM = {}
results_efficiency_CB = {}

for id in df_plot["id"].unique():

    filename = "Results/efficiency/{}_efficiency.txt".format(id)
    filename_ER = "Results/efficiency/null_models/ER/{}_efficiency.txt".format(id)
    filename_CM = "Results/efficiency/null_models/CM/{}_efficiency.txt".format(id)
    filename_CB = "Results/efficiency/null_models/CB/{}_efficiency.txt".format(id)

    τs, entropy_arr, free_energy_arr, ηs_arr = np.loadtxt(filename, unpack=True)

    τs_ER, entropy_ER, free_energy_ER, ηs_ER, entropy_ER_std, free_energy_ER_std, ηs_ER_std = np.loadtxt(filename_ER, unpack=True)
    τs_CM, entropy_CM, free_energy_CM, ηs_CM, entropy_CM_std, free_energy_CM_std, ηs_CM_std = np.loadtxt(filename_CM, unpack=True)
    τs_CB, entropy_CB, free_energy_CB, ηs_CB, entropy_CB_std, free_energy_CB_std, ηs_CB_std = np.loadtxt(filename_CB, unpack=True)

    results_efficiency[id] = {
        "tau": τs,
        "entropy": entropy_arr,
        "free_energy": free_energy_arr,
        "network_efficiency": ηs_arr,
    }

    results_efficiency_ER[id] = {
        "tau": τs_ER,
        "entropy": entropy_ER,
        "free_energy": free_energy_ER,
        "network_efficiency": ηs_ER,
        "entropy_std": entropy_ER_std,
        "free_energy_std": free_energy_ER_std,
        "network_efficiency_std": ηs_ER_std,
    }

    results_efficiency_CM[id] = {
        "tau": τs_CM,
        "entropy": entropy_CM,
        "free_energy": free_energy_CM,
        "network_efficiency": ηs_CM,
        "entropy_std": entropy_CM_std,
        "free_energy_std": free_energy_CM_std,
        "network_efficiency_std": ηs_CM_std,
    }

    results_efficiency_CB[id] = {
        "tau": τs_CB,
        "entropy": entropy_CB,
        "free_energy": free_energy_CB,
        "network_efficiency": ηs_CB,
        "entropy_std": entropy_CB_std,
        "free_energy_std": free_energy_CB_std,
        "network_efficiency_std": ηs_CB_std,
    }
        

# Compute efficiency at τ=τ_diff for each network
for i, id in enumerate(df_plot["id"].unique()):
    if id not in results_efficiency:
        continue

    τs = results_efficiency[id]["tau"]
    ηs_arr = results_efficiency[id]["network_efficiency"]

    τs_ER = results_efficiency_ER[id]["tau"]
    ηs_ER = results_efficiency_ER[id]["network_efficiency"]

    τs_CM = results_efficiency_CM[id]["tau"]
    ηs_CM = results_efficiency_CM[id]["network_efficiency"]

    τs_CB = results_efficiency_CB[id]["tau"]
    ηs_CB = results_efficiency_CB[id]["network_efficiency"]

    efficiency_at_τ_diff = np.interp(
        df_plot[df_plot["id"] == id]["diffusion_time"].values[0], τs, ηs_arr
    )
    efficiency_at_τ_diff_ER = np.interp(
        df_plot[df_plot["id"] == id]["diffusion_time"].values[0], τs_ER, ηs_ER
    )
    efficiency_at_τ_diff_CM = np.interp(
        df_plot[df_plot["id"] == id]["diffusion_time"].values[0], τs_CM, ηs_CM
    )
    efficiency_at_τ_diff_CB = np.interp(
        df_plot[df_plot["id"] == id]["diffusion_time"].values[0], τs_CB, ηs_CB
    )

    df_plot.loc[df_plot["id"] == id, "efficiency_at_τ_diff"] = efficiency_at_τ_diff
    df_plot.loc[df_plot["id"] == id, "efficiency_at_τ_diff_ER"] = efficiency_at_τ_diff_ER
    df_plot.loc[df_plot["id"] == id, "efficiency_at_τ_diff_CM"] = efficiency_at_τ_diff_CM
    df_plot.loc[df_plot["id"] == id, "efficiency_at_τ_diff_CB"] = efficiency_at_τ_diff_CB

df_plot["efficiency_diff_ER"] = (
    df_plot["efficiency_at_τ_diff"] - df_plot["efficiency_at_τ_diff_ER"]
)
df_plot["efficiency_diff_CM"] = (
    df_plot["efficiency_at_τ_diff"] - df_plot["efficiency_at_τ_diff_CM"]
)
df_plot["efficiency_diff_CB"] = (
    df_plot["efficiency_at_τ_diff"] - df_plot["efficiency_at_τ_diff_CB"]
)

In [ ]:
df_plot_efficiency = df_plot[df_plot["S"] > 5]

fig, ax = plt.subplot_mosaic(
    """
    AB
    CD
    """,
    figsize=(8 * 2, 6*2),
    gridspec_kw={"hspace": 0.3, "wspace": 0.3},
)

letters = ["A", "B", "C", "D"]

mean_efficiencies = []
mean_entropies = []
mean_free_energies = []

for n_type in df_plot_efficiency["network_type"].unique():
    
    df_subset = df_plot_efficiency[df_plot_efficiency["network_type"] == n_type]
    ids = df_subset["id"].values.astype(str)
    label = n_type.replace("_", " ").title()
    color = color_dict.get(n_type, "gray")

    mean_efficiency = np.zeros((len(ids), len(τs)))
    mean_entropy = np.zeros((len(ids), len(τs)))
    mean_free_energy = np.zeros((len(ids), len(τs)))

    for i, id in enumerate(ids):

        if id not in results_efficiency:
            continue

        τs = results_efficiency[id]["tau"]
        entropy_arr = results_efficiency[id]["entropy"]
        free_energy_arr = results_efficiency[id]["free_energy"]
        ηs_arr = results_efficiency[id]["network_efficiency"]

        mean_efficiency[i, :] = ηs_arr
        mean_entropy[i, :] = entropy_arr
        mean_free_energy[i, :] = free_energy_arr

    mean_efficiency = np.nanmean(mean_efficiency, axis=0)
    mean_entropy = np.nanmean(mean_entropy, axis=0) 
    mean_free_energy = np.nanmean(mean_free_energy, axis=0)

    mean_efficiencies.append(mean_efficiency)
    mean_entropies.append(mean_entropy)
    mean_free_energies.append(mean_free_energy)
    
    # Standar error of the mean
    se_mean_efficiency = np.nanstd(mean_efficiency) / np.sqrt(len(ids))

    ax["A"].plot(τs, mean_efficiency, color=color, label=label, lw=3)
    ax["A"].fill_between(τs, mean_efficiency - se_mean_efficiency, mean_efficiency + se_mean_efficiency, color=color, alpha=0.2)

ax["A"].set_xscale("log")

ax["A"].set_xlabel(r"Propagation scale ($\tau$)", fontsize=24)
ax["A"].set_ylabel(r"Mean efficiency ($\langle \eta \rangle$)", fontsize=24)

handles = []
labels = []

for n_type, color in color_dict.items():
    # Use patch for legend
    handles.append(Patch(color=color))
    labels.append("{} ({})".format(n_type.replace("_", " ").title(), net_type_initials[n_type]))

ax["A"].legend(handles, labels, fontsize=14, ncol=3, bbox_to_anchor=(1.0, 1.30), loc="upper center")

# Violinplot of efficiency difference for ER null model
sns.violinplot(
    x="network_type",
    y="efficiency_diff_ER",
    data=df_plot_efficiency,
    palette=[
        color_dict.get(n_type, "gray") for n_type in df_plot_efficiency["network_type"].unique()
    ],
    ax=ax["B"],
)
ax["B"].axhline(0, color="k", ls="--", lw=2)

ax["B"].set_xticklabels(
    [net_type_initials[n_type] for n_type in df_plot_efficiency["network_type"].unique()],
)

ax["B"].set_xlabel("", fontsize=24)
ax["B"].set_ylabel(r"Efficiency difference ($\Delta \eta_{ER}$)", fontsize=24)
ax["B"].tick_params(axis="y", which="major", labelsize=14)

# Violinplot of efficiency difference for CM null model
sns.violinplot(
    x="network_type",
    y="efficiency_diff_CM",
    data=df_plot_efficiency,
    palette=[
        color_dict.get(n_type, "gray") for n_type in df_plot_efficiency["network_type"].unique()
    ],
    ax=ax["C"],
)
ax["C"].axhline(0, color="k", ls="--", lw=2)

ax["C"].set_xticklabels(
    [net_type_initials[n_type] for n_type in df_plot_efficiency["network_type"].unique()],
)

ax["C"].set_xlabel("", fontsize=24)
ax["C"].set_ylabel(r"Efficiency difference ($\Delta \eta_{CM}$)", fontsize=24)
ax["C"].tick_params(axis="y", which="major", labelsize=14)

# Difference in efficiency with perturbations
# Use kde
sns.kdeplot(
    df_perturbations["mean_diff_more_links"],
    ax=ax["D"],
    label="More links",
    color="#2B2B2B",
    fill=True,
    alpha=0.5,
)
sns.kdeplot(
    df_perturbations["mean_diff_less_links"],
    ax=ax["D"],
    label="Less links",
    color="plum",
    fill=True,
    alpha=0.7,
)

ax["D"].set_xlabel(r"Efficiency difference ($\Delta \eta$)", fontsize=24)
ax["D"].set_ylabel("Density", fontsize=24)  

ax["D"].legend(fontsize=14)

# Inset with proportion of negative differences
ax_inset = ax["D"].inset_axes([0.15, 0.6, 0.3, 0.35])

prop_more_links = (df_perturbations["mean_diff_more_links"] < 0).mean()
prop_less_links = (df_perturbations["mean_diff_less_links"] < 0).mean()

prop_more_std_err = proportion_confint(
    (df_perturbations["mean_diff_more_links"] < 0).sum(), len(df_perturbations)
)
prop_less_std_err = proportion_confint(
    (df_perturbations["mean_diff_less_links"] < 0).sum(), len(df_perturbations)
)

ax_inset.bar(
    [0, 1],
    [prop_more_links, prop_less_links],
    yerr=[
        prop_more_std_err[1] - prop_more_links,
        prop_less_std_err[1] - prop_less_links,
    ],
    color=["#2B2B2B", "plum"],
    alpha=0.7,
    capsize=5,
)

ax_inset.set_xticks([0, 1])
ax_inset.set_xticklabels(["More links", "Less links"], rotation=90, fontsize=12)
ax_inset.set_ylabel(r"$\mathrm{Pr}(\Delta \eta < 0)$", fontsize=14)
ax_inset.set_ylim(0, 1)

for letter in letters:
    ax[letter].tick_params(axis="both", which="major", labelsize=16)
    ax[letter].tick_params(axis="both", which="minor", labelsize=16)

    # Add subplot labels
    if (letter == "C") or (letter == "D"):
        ax[letter].text(
            0.85,
            0.15,
            "({})".format(letter.lower()),
            transform=ax[letter].transAxes,
            fontsize=24,
            fontweight="bold",
            va="top",
        )

    else:
        ax[letter].text(
            0.85,
            0.95,
            "({})".format(letter.lower()),
            transform=ax[letter].transAxes,
            fontsize=24,
            fontweight="bold",
            va="top",
        )

In [ ]:
null = "CM"

df_plot_efficiency["more_efficiency_CM"] = (df_plot_efficiency["efficiency_diff_CM"] > 0).astype(int)

df_null_models = pd.DataFrame()

for i, id in enumerate(df_plot_efficiency["id"].unique()):

    df_i = pd.read_csv(f"Results/metrics/null_models/{null}/{id}_metrics.csv")
    df_null_models = pd.concat([df_null_models, df_i], axis=0)

df_comp = df_plot_efficiency.merge(df_null_models, on="id", suffixes=("", "_null"))

# Take only plant-pollinator, anemone-fish, seed-dispersal, plant-ant
df_comp = df_comp[df_comp["network_type"].isin(["anemone_fish", "plant_ant", "plant_pollinator", "seed_dispersal"])]

#########
fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(8 * 2, 6),
    gridspec_kw={"wspace":0.25}
)

letters = ["A", "B"]

# Bar plot of the proportion of networks with higher efficiency than CM null model
for n_type in df_plot_efficiency["network_type"].unique():

    subset = df_plot_efficiency[df_plot_efficiency["network_type"] == n_type]

    prop_more_efficiency = subset["more_efficiency_CM"].mean()
    ci_low, ci_upp = proportion_confint(subset["more_efficiency_CM"].sum(), len(subset))

    ax["A"].bar(
        n_type,
        prop_more_efficiency,
        yerr=[[prop_more_efficiency - ci_low], [ci_upp - prop_more_efficiency]],
        color=color_dict.get(n_type, "gray"),
        capsize=5,
    )

ax["A"].set_xticklabels(
    [net_type_initials[n_type] for n_type in df_plot_efficiency["network_type"].unique()],
    ha="right",
    fontsize=14,
)

ax["A"].tick_params(axis="y", which="major", labelsize=14)

#ax["A"].set_ylabel(r"$\mathrm{Pr}(\Delta \eta_{CM})>0$", fontsize=26)

# In words
ax["A"].set_ylabel(r"Fraction of networks with $\Delta \eta_{CM} > 0$", fontsize=24)

# B
x = df_comp["efficiency_diff_CM"].values
y = df_comp["global_efficiency"] - df_comp["global_efficiency_mean"]

lr = linregress(x, y)

p_val = lr.pvalue

if p_val < 0.05:
    label_pval = "p < 0.001"

xx = np.linspace(min(x), max(x), 100)

for n_type in df_comp["network_type"].unique():

    subset = df_comp[df_comp["network_type"] == n_type]

    ax["B"].scatter(
        subset["efficiency_diff_CM"],
        subset["global_efficiency"] - subset["global_efficiency_mean"],
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
    )

ax["B"].plot(xx, lr.intercept + lr.slope * xx, color="k", lw=3, ls="-", label=r"$R^2={:.2f}$, ${}$".format(lr.rvalue**2, label_pval))

ax["B"].set_xlabel(r"$\Delta \eta_{CM}$", fontsize=30)
ax["B"].set_ylabel(r"$\Delta \epsilon_{CM}$", fontsize=30)

ax["B"].tick_params(axis="y", which="major", labelsize=14)

ax["B"].legend(fontsize=14, loc="lower right")

handles = []
labels = []

for n_type, color in color_dict.items():
    handles.append(Patch(color=color, label=n_type.replace("_", " ").title()))
    labels.append("{} ({})".format(n_type.replace("_", " ").title(), net_type_initials[n_type]))
    
# Label subplots
for letter in letters:
    ax[letter].text(
        0.05,
        0.95,
        "({})".format(letter.lower()),
        transform=ax[letter].transAxes,
        fontsize=24,
        fontweight="bold",
        va="top",
    )

fig.legend(handles, labels, fontsize=14, ncol=3, bbox_to_anchor=(0.7, 1.05))

# SI

## C vs S separated by network type

In [ ]:
# Plot of linear regression for each network type, with slope and r^2 in the legend
fig, ax = plt.subplot_mosaic(
    """
    AB
    CD
    EF
    """,
    figsize=(8 * 2, 6 * 3),
)

letters = ["A", "B", "C", "D", "E", "F"]

slopes = []

k = 0
for n_type in df_plot["network_type"].unique():
    subset = df_plot[df_plot["network_type"] == n_type]

    x = subset["S"].values.astype(float)
    y = subset["C"].values.astype(float)

    res = linregress(np.log(x), np.log(y))

    r_squared = res.rvalue**2
    p_value = res.pvalue

    ax[letters[k]].scatter(
        df_plot["S"],
        df_plot["C"],
        s=20,
        color="lightgray",
        alpha=0.5,
    )

    ax[letters[k]].scatter(
        x,
        y,
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
        label=f"{n_type.replace('_', ' ').title()} (slope={res.slope:.2f}, $R^2$={r_squared:.2f})",
    )

    x_fit = np.logspace(0, 3, 100)
    y_fit = np.exp(res.intercept) * np.power(x_fit, res.slope)

    ax[letters[k]].plot(x_fit, y_fit, lw=3, color="k", ls="--")

    ax[letters[k]].set_xscale("log")
    ax[letters[k]].set_yscale("log")

    ax[letters[k]].set_xlabel(r"$S$", fontsize=24)
    ax[letters[k]].set_ylabel(r"$C$", fontsize=24)

    # ax[letters[k]].text(
    #     0.05,
    #     0.3,
    #     r"$\alpha = {:.2f}$".format(res.slope),
    #     transform=ax[letters[k]].transAxes,
    #     fontsize=12,
    # )

    CI = 1.96 * res.stderr
    
    exponent_upper = res.slope + CI
    exponent_lower = res.slope - CI
    
    ax[letters[k]].text(
        0.05,
        0.3,
        r"$\alpha = {:.2f}$ $[{:.2f}, {:.2f}]$".format(res.slope, exponent_lower, exponent_upper),
        transform=ax[letters[k]].transAxes,
        fontsize=12,
    )

    ax[letters[k]].text(
        0.05,
        0.2,
        r"$R^2 = {:.2f}$".format(r_squared),
        transform=ax[letters[k]].transAxes,
        fontsize=12,
    )

    ax[letters[k]].text(
        0.05,
        0.1,
        r"$p = {:.1e}$".format(p_value),
        transform=ax[letters[k]].transAxes,
        fontsize=12,
    )

    slopes.append(res.slope)

    k += 1

handles = []
labels = []

for n_type, color in color_dict.items():
    handles.append(
        plt.Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=color, markersize=10
        )
    )
    labels.append(n_type.replace("_", " ").title())

fig.legend(
    handles, labels, loc="upper center", ncol=4, fontsize=14, bbox_to_anchor=(0.5, 0.93)
)

# Label subplots
for letter in letters:
    ax[letter].tick_params(axis="both", which="major", labelsize=16)
    ax[letter].tick_params(axis="both", which="minor", labelsize=16)

    ax[letter].text(
        0.85,
        0.85,
        "({})".format(letter.lower()),
        transform=ax[letter].transAxes,
        fontsize=24,
        fontweight="bold",
    )

# C vs S separating by bipartite and non-bipartite

In [ ]:
# C vs S separated by bipartite and non-bipartite networks
fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(8 * 2, 6),
)

letters = ["A", "B"]

df_bipartite = df_plot[df_plot["is_bipartite"] == True]
df_non_bipartite = df_plot[df_plot["is_bipartite"] == False]

for letter, df_subset in zip(letters, [df_bipartite, df_non_bipartite]):
    
    ax[letter].scatter(
        df_subset["S"],
        df_subset["C"],
        alpha=1,
        s=20,
    )

    x = np.logspace(0, 3, 100)
    y = SC * np.power(x, -1)
    
    fit = linregress(np.log(df_subset["S"].values.astype(float)), np.log(df_subset["C"].values.astype(float)))

    ax[letter].plot(x, y, lw=3, color="k", label=r"$C \sim S^{-1}$")
    ax[letter].plot(x, np.exp(fit.intercept) * np.power(x, fit.slope), lw=3, color="k", ls="--", label=r"Fit: $C \sim S^{%.2f}$" % fit.slope)
    
    # Plot as text R^2, pvalue and exponent +- 95% CI
    CI = 1.96 * fit.stderr
    
    exponent_upper = fit.slope + CI
    exponent_lower = fit.slope - CI
    
    ax[letter].text(
        0.05,
        0.3,
        r"$\alpha = {:.2f}$ $[{:.2f}, {:.2f}]$".format(fit.slope, exponent_lower, exponent_upper),
        transform=ax[letter].transAxes,
        fontsize=12,
    )
    ax[letter].text(
        0.05,
        0.2,
        r"$R^2 = {:.2f}$".format(fit.rvalue**2),
        transform=ax[letter].transAxes,
        fontsize=12,
    )
        
    ax[letter].text(
        0.05,
        0.1,
        r"$p = {:.1e}$".format(fit.pvalue),
        transform=ax[letter].transAxes,
        fontsize=12,
    )

    ax[letter].set_xscale("log")
    ax[letter].set_yscale("log")

    ax[letter].set_xlabel(r"$S$", fontsize=24)
    ax[letter].set_ylabel(r"$C$", fontsize=24)

    ax[letter].legend(fontsize=14)
    
    ax[letter].set_title("Bipartite" if letter == "A" else "Non-bipartite", fontsize=18)

## Comparison with NatComm data

In [ ]:
# C vs S for our dataset plus the NatComm data
plt.figure(figsize=(8, 6))

plt.scatter(
    df_plot["S"], df_plot["C"], label="Our dataset", color="lightgray", marker="o", s=20
)

plt.scatter(
    df_natcomm["S"],
    df_natcomm["C"],
    label="Jacquet et al. (2016)",
    color="purple",
    marker="o",
    s=20,
)

tot_S = (
    df_plot["S"].values.astype(float).tolist()
    + df_natcomm["S"].values.astype(float).tolist()
)
tot_C = (
    df_plot["C"].values.astype(float).tolist()
    + df_natcomm["C"].values.astype(float).tolist()
)

reg = linregress(np.log(tot_S), np.log(tot_C))

x = np.logspace(0, 3, 100)
y = SC * np.power(x, -1)

plt.plot(x, y, lw=3, color="k", label=r"$C \sim S^{-1}$")
plt.plot(
    x,
    np.exp(reg.intercept) * np.power(x, reg.slope),
    lw=3,
    color="k",
    ls="--",
    label=r"Data fit: $C \sim S^{%.2f}$" % reg.slope,
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel(r"$S$", fontsize=24)
plt.ylabel(r"$C$", fontsize=24)

plt.legend(fontsize=14, ncol=2, bbox_to_anchor=(0.95, 1.2))

## Average degree vs species richness for each network type

In [ ]:
# Plot of avg degree vs S for each network type with the linear regression slope and R^2 in the legend
fig, ax = plt.subplot_mosaic(
    """
    AB
    CD
    EF
    """,
    figsize=(8 * 2, 6 * 3),
    #sharey=True
)

letters = ["A", "B", "C", "D", "E", "F"]

for i, n_type in enumerate(df_plot["network_type"].unique()):
    subset = df_plot[df_plot["network_type"] == n_type]

    x = subset["S"].values.astype(float)
    y = subset["mean_degree"].values.astype(float)

    res = linregress(np.log(x), np.log(y))

    r_squared = res.rvalue**2
    p_value = res.pvalue

    ax[letters[i]].scatter(
        x,
        y,
        color=color_dict.get(n_type, "gray"),
        marker="o",
        s=20,
        label=f"{n_type.replace('_', ' ').title()} (slope={res.slope:.2f}, $R^2$={r_squared:.2f})",
    )

    x_fit = np.logspace(0, 3, 100)
    y_fit = np.exp(res.intercept) * np.power(x_fit, res.slope)

    ax[letters[i]].plot(x_fit, y_fit, lw=3, color="k", ls="--")

    ax[letters[i]].set_xscale("log")
    ax[letters[i]].set_yscale("log")

    ax[letters[i]].set_xlabel(r"$S$", fontsize=24)
    ax[letters[i]].set_ylabel(r"$\langle k \rangle$", fontsize=24)

    ax[letters[i]].text(
        0.05,
        0.7,
        r"$\langle k \rangle \sim S^{{{:.2f}}}$".format(res.slope),
        transform=ax[letters[i]].transAxes,
        fontsize=16,
    )

    ax[letters[i]].text(
        0.05,
        0.6,
        r"$R^2 = {:.2f}$".format(r_squared),
        transform=ax[letters[i]].transAxes,
        fontsize=14,
    )
    
    ax[letters[i]].text(
        0.05,
        0.5,
        r"$p = {:.1e}$".format(p_value),
        transform=ax[letters[i]].transAxes,
        fontsize=14,
    )
    
handles = []
labels = []

for n_type, color in color_dict.items():
    handles.append(
        plt.Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=color, markersize=10
        )
    )
    labels.append(n_type.replace("_", " ").title())
    
fig.legend(
    handles, labels, loc="upper center", ncol=3, fontsize=14, bbox_to_anchor=(0.5, 0.93)
)

# Label subplots
for letter in letters:
    ax[letter].tick_params(axis="both", which="major", labelsize=16)
    ax[letter].tick_params(axis="both", which="minor", labelsize=16)
    
    ax[letter].set_ylim(0.5,110)
    
    ax[letter].text(
        0.05,
        0.9,
        "({})".format(letter.lower()),
        transform=ax[letter].transAxes,
        fontsize=24,
        fontweight="bold",
    )

## Degree distribution

In [ ]:
# Degree distribution
degrees = []

for id in df_plot["id"].unique():

    filename = f"Results/degree_dist/degree_distribution_{id}.csv"

    try:

        df = pd.read_csv(filename)

        degrees.extend(df["degree"].values.tolist())

    except:
        pass

degrees = np.array(degrees)

fit = powerlaw.Fit(degrees, xmin=1)

fig, ax = plt.subplot_mosaic(
    """
    C
    """,
    figsize=(8, 6),
)

fit.plot_pdf(
    ax=ax["C"],
    color="salmon",
    ls="",
    marker="o",
    markersize=10,
)

fit.truncated_power_law.plot_pdf(
    ax=ax["C"],
    color="k",
    ls="--",
    lw=3,
    label=r"$P(k) \sim k^{-%.2f} e^{-k/%.2f}$"
    % (fit.truncated_power_law.alpha, fit.truncated_power_law.Lambda),
)

ax["C"].set_xlabel(r"$k$", fontsize=24)
ax["C"].set_ylabel("Prob. density", fontsize=20)

ax["C"].tick_params(axis="both", which="major", labelsize=16)

ax["C"].legend(fontsize=16)

## Efficiency

In [ ]:
fig, ax = plt.subplot_mosaic(
    """
    A
    """,
    figsize=(8, 6),
)

for n_type in df_plot_efficiency["network_type"].unique():

    df_subset = df_plot_efficiency[df_plot_efficiency["network_type"] == n_type]
    ids = df_subset["id"].values.astype(str)
    label = n_type.replace("_", " ").title()
    color = color_dict.get(n_type, "gray")

    for i, id in enumerate(ids):

        if id not in results_efficiency:
            continue

        τs = results_efficiency[id]["tau"]
        entropy_arr = results_efficiency[id]["entropy"]
        free_energy_arr = results_efficiency[id]["free_energy"]
        ηs_arr = results_efficiency[id]["network_efficiency"]

        ax["A"].plot(τs, ηs_arr, color=color, alpha=0.5)
        
ax["A"].set_xscale("log")
ax["A"].set_xlabel(r"$\tau$", fontsize=30)
ax["A"].set_ylabel(r"$\eta$", fontsize=30)

ax["A"].tick_params(axis="both", which="major", labelsize=14)

handles = []
labels = []

for n_type, color in color_dict.items():
    handles.append(
        plt.Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=color, markersize=10
        )
    )
    labels.append(n_type.replace("_", " ").title())

ax["A"].legend(handles=handles, labels=labels, fontsize=14, ncol=3, bbox_to_anchor=(1.05, 1.25))

## Efficiency comparison to null models

In [ ]:
fig, ax = plt.subplot_mosaic(
    """
    AB
    CD
    EF
    GH
    """,
    figsize=(8 * 2, 6 * 4),
    # sharex=True,
    # sharey=True,
)

letters = ["A", "B", "C", "D", "E", "F", "G", "H"]

k = 0
for n_type in df_plot_efficiency["network_type"].unique():
    
    df_subset = df_plot_efficiency[df_plot_efficiency["network_type"] == n_type]
    ids = df_subset["id"].values.astype(str)
    label = n_type.replace("_", " ").title()
    color = color_dict.get(n_type, "gray")

    mean_efficiency = np.zeros((len(ids), len(τs)))
    mean_entropy = np.zeros((len(ids), len(τs)))
    mean_free_energy = np.zeros((len(ids), len(τs)))

    mean_efficiency_ER = np.zeros((len(ids), len(τs_ER)))
    mean_entropy_ER = np.zeros((len(ids), len(τs_ER)))
    mean_free_energy_ER = np.zeros((len(ids), len(τs_ER)))

    std_efficiency_ER = np.zeros((len(ids), len(τs_ER)))
    std_entropy_ER = np.zeros((len(ids), len(τs_ER)))
    std_free_energy_ER = np.zeros((len(ids), len(τs_ER)))

    mean_efficiency_CM = np.zeros((len(ids), len(τs_CM)))
    mean_entropy_CM = np.zeros((len(ids), len(τs_CM)))
    mean_free_energy_CM = np.zeros((len(ids), len(τs_CM)))

    std_efficiency_CM = np.zeros((len(ids), len(τs_CM)))
    std_entropy_CM = np.zeros((len(ids), len(τs_CM)))
    std_free_energy_CM = np.zeros((len(ids), len(τs_CM)))

    for i, id in enumerate(ids):

        if id not in results_efficiency:
            continue

        # Empirical
        τs = results_efficiency[id]["tau"]
        entropy_arr = results_efficiency[id]["entropy"]
        free_energy_arr = results_efficiency[id]["free_energy"]
        ηs_arr = results_efficiency[id]["network_efficiency"]

        # ER null model
        τs_ER = results_efficiency_ER[id]["tau"]
        entropy_ER = results_efficiency_ER[id]["entropy"]
        free_energy_ER = results_efficiency_ER[id]["free_energy"]
        ηs_ER = results_efficiency_ER[id]["network_efficiency"]

        # CM null model
        τs_CM = results_efficiency_CM[id]["tau"]
        entropy_CM = results_efficiency_CM[id]["entropy"]
        free_energy_CM = results_efficiency_CM[id]["free_energy"]
        ηs_CM = results_efficiency_CM[id]["network_efficiency"]

        # Store results in arrays for mean and std calculation
        mean_efficiency[i, :] = ηs_arr
        mean_entropy[i, :] = entropy_arr
        mean_free_energy[i, :] = free_energy_arr

        mean_efficiency_ER[i, :] = ηs_ER
        mean_entropy_ER[i, :] = entropy_ER
        mean_free_energy_ER[i, :] = free_energy_ER

        mean_efficiency_CM[i, :] = ηs_CM
        mean_entropy_CM[i, :] = entropy_CM
        mean_free_energy_CM[i, :] = free_energy_CM
        
        std_efficiency_ER[i, :] = results_efficiency_ER[id]["network_efficiency_std"]
        std_entropy_ER[i, :] = results_efficiency_ER[id]["entropy_std"]
        std_free_energy_ER[i, :] = results_efficiency_ER[id]["free_energy_std"]
        
        std_efficiency_CM[i, :] = results_efficiency_CM[id]["network_efficiency_std"]
        std_entropy_CM[i, :] = results_efficiency_CM[id]["entropy_std"]
        std_free_energy_CM[i, :] = results_efficiency_CM[id]["free_energy_std"]

    mean_efficiency = np.nanmean(mean_efficiency, axis=0)
    mean_entropy = np.nanmean(mean_entropy, axis=0)
    mean_free_energy = np.nanmean(mean_free_energy, axis=0)

    mean_efficiency_ER = np.nanmean(mean_efficiency_ER, axis=0)
    mean_entropy_ER = np.nanmean(mean_entropy_ER, axis=0)
    mean_free_energy_ER = np.nanmean(mean_free_energy_ER, axis=0)

    mean_efficiency_CM = np.nanmean(mean_efficiency_CM, axis=0)
    mean_entropy_CM = np.nanmean(mean_entropy_CM, axis=0)
    mean_free_energy_CM = np.nanmean(mean_free_energy_CM, axis=0)
    
    mean_std_efficiency_ER = np.nanmean(std_efficiency_ER, axis=0)
    mean_std_entropy_ER = np.nanmean(std_entropy_ER, axis=0)
    mean_std_free_energy_ER = np.nanmean(std_free_energy_ER, axis=0)
    
    mean_std_efficiency_CM = np.nanmean(std_efficiency_CM, axis=0)
    mean_std_entropy_CM = np.nanmean(std_entropy_CM, axis=0)
    mean_std_free_energy_CM = np.nanmean(std_free_energy_CM, axis=0)

    # Plot mean efficiency for this group in the corresponding panel
    ax[letters[k]].plot(τs, mean_efficiency, color=color, label=label, lw=4)
    ax[letters[k]].plot(τs, mean_efficiency_ER, color="black", label="ER", lw=4)
    ax[letters[k]].plot(τs, mean_efficiency_CM, color="gray", label="CM", lw=4)

    # ax[letters[k]].fill_between(τs, mean_efficiency_ER - mean_std_efficiency_ER, mean_efficiency_ER + mean_std_efficiency_ER, color="black", alpha=0.2)
    # ax[letters[k]].fill_between(τs, mean_efficiency_CM - mean_std_efficiency_CM, mean_efficiency_CM + mean_std_efficiency_CM, color="gray", alpha=0.2)

    ax[letters[k]].set_xscale("log")
    ax[letters[k]].legend(fontsize=14)
    
    ax[letters[k]].set_xlabel(r"$\tau$", fontsize=30)
    ax[letters[k]].set_ylabel(r"$\langle \eta \rangle$", fontsize=30)
    
    ax[letters[k]].tick_params(axis="both", which="major", labelsize=16)

    k += 1

# Boxplot of typical diffusion time for each network type
sns.boxplot(x="network_type", y="diffusion_time", data=df_plot_efficiency, ax=ax["G"], palette=[color_dict.get(n_type, "gray") for n_type in df_plot_efficiency["network_type"].unique()])

ax["G"].set_xticklabels([n_type.replace("_", " ").title() for n_type in df_plot_efficiency["network_type"].unique()], rotation=90, ha="right", fontsize=14)

ax["G"].set_yscale("log")

ax["G"].set_xlabel("", fontsize=30)
ax["G"].set_ylabel(r"$\tau_{diff}$", fontsize=30)

ax["G"].tick_params(axis="y", which="major", labelsize=16)

# Boxplot of efficiency at diffusion time for each network type
sns.boxplot(x="network_type", y="efficiency_at_τ_diff", data=df_plot_efficiency, ax=ax["H"], palette=[color_dict.get(n_type, "gray") for n_type in df_plot_efficiency["network_type"].unique()])

ax["H"].set_xticklabels([n_type.replace("_", " ").title() for n_type in df_plot_efficiency["network_type"].unique()], rotation=90, ha="right", fontsize=14)
#ax["H"].set_yscale("log")

ax["H"].set_xlabel("", fontsize=30)
ax["H"].set_ylabel(r"$\eta(\tau_{diff})$", fontsize=30)

ax["H"].tick_params(axis="y", which="major", labelsize=16)

In [ ]:
df_perturbations_percentage = pd.DataFrame()

for percentage in range(1, 11):
    
    print("\rProcessing percentage: {}%".format(percentage), end="")

    filenames = glob.glob(f"Results/efficiency/perturbations/percentage_{percentage}/*.csv")

    for filename in filenames:
        df_i = pd.read_csv(filename)

        df_perturbations_percentage = pd.concat([df_perturbations_percentage, df_i], ignore_index=True)
        
# Compute mean and std of the differences for each percentage
df_perturbations_percentage_avg = df_perturbations_percentage.groupby("percentage").agg(
    mean_diff_more_links=("mean_diff_more_links", "mean"),
    std_diff_more_links=("mean_diff_more_links", "sem"),
    mean_diff_less_links=("mean_diff_less_links", "mean"),
    std_diff_less_links=("mean_diff_less_links", "sem"),
    median_diff_more_links=("mean_diff_more_links", "median"),
    median_diff_less_links=("mean_diff_less_links", "median"),
).reset_index()

# Rewrite the dataframe to plot split violins with type of perturbation (more links, less links) as hue
df_perturbations_melted = df_perturbations_percentage.melt(
    id_vars=["percentage"],
    value_vars=["mean_diff_more_links", "mean_diff_less_links"],
    var_name="perturbation_type",
    value_name="efficiency_difference",
)

# Map perturbation_type to more readable labels
df_perturbations_melted["perturbation_type"] = df_perturbations_melted["perturbation_type"].map({
    "mean_diff_more_links": "More links",
    "mean_diff_less_links": "Less links"
})

# Plot the difference in efficiency with perturbations for each percentage of links added/removed with error bars
fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(8 * 2, 6),
)

# Mean
ax["A"].errorbar(
    df_perturbations_percentage_avg["percentage"],
    df_perturbations_percentage_avg["mean_diff_more_links"],
    # yerr=df_perturbations_percentage_avg["std_diff_more_links"],
    # 95% confidence interval
    yerr=1.96 * df_perturbations_percentage_avg["std_diff_more_links"],
    label="Mean",
    color="darkseagreen",
    lw=5,
    elinewidth=1,
    capsize=5,
)
ax["A"].errorbar(
    -df_perturbations_percentage_avg["percentage"],
    df_perturbations_percentage_avg["mean_diff_less_links"],
    # yerr=df_perturbations_percentage_avg["std_diff_less_links"],
    # 95% confidence interval
    yerr=1.96 * df_perturbations_percentage_avg["std_diff_less_links"],
    color="darkseagreen",
    lw=5,
    elinewidth=1,
    capsize=5,
)

# Median
ax["A"].errorbar(
    df_perturbations_percentage_avg["percentage"],
    df_perturbations_percentage_avg["median_diff_more_links"],
    yerr=1.96 *df_perturbations_percentage_avg["std_diff_more_links"],
    label="Median",
    color="steelblue",
    lw=5,
    elinewidth=1,
    capsize=5,
)

ax["A"].errorbar(
    -df_perturbations_percentage_avg["percentage"],
    df_perturbations_percentage_avg["median_diff_less_links"],
    yerr=1.96 * df_perturbations_percentage_avg["std_diff_less_links"],
    color="steelblue",
    lw=5,
    elinewidth=1,
    capsize=5,
)

ax["A"].axhline(0, color="k", ls="--", lw=2)

ax["A"].set_xlabel(r"Links perturbed", fontsize=30)
ax["A"].set_ylabel(r"$\Delta \eta$", fontsize=30)

tick_labels = ["{}%".format(p) for p in range(-10, 11, 2)]
tick_positions = [p for p in range(-10, 11, 2)]

ax["A"].set_xticks(tick_positions)
ax["A"].set_xticklabels(tick_labels, fontsize=14)

ax["A"].legend(fontsize=14)

# Violin plots of the differences for each percentage
sns.violinplot(
    x="percentage",
    y="efficiency_difference",
    hue="perturbation_type",
    data=df_perturbations_melted,
    palette={"More links": "plum", "Less links": "lightblue"},
    split=True,
    inner="quart",
    ax=ax["B"],
)

ax["B"].axhline(0, color="k", ls="--", lw=2)

ax["B"].set_xlabel(r"Links perturbed", fontsize=30)
ax["B"].set_ylabel(r"$\Delta \eta$", fontsize=30)

tick_labels = ["{}%".format(p) for p in range(1, 11, 1)]

ax["B"].set_xticklabels(tick_labels, fontsize=14)

In [ ]:
df_plot_efficiency_bipartite = df_plot_efficiency[df_plot_efficiency["is_bipartite"] == True]

fig, ax = plt.subplot_mosaic(
    """
    AB
    """,
    figsize=(8 * 2, 6),
    gridspec_kw={"wspace":0.25}
)

# Violinplot of efficiency difference for CM null model
sns.violinplot(
    x="network_type",
    y="efficiency_diff_CM",
    data=df_plot_efficiency_bipartite,
    palette=[
        color_dict.get(n_type, "gray") for n_type in df_plot_efficiency_bipartite["network_type"].unique()
    ],
    ax=ax["A"],
)
ax["A"].axhline(0, color="k", ls="--", lw=2)

ax["A"].set_xticklabels(
    [net_type_initials[n_type] for n_type in df_plot_efficiency_bipartite["network_type"].unique()],
)

ax["A"].set_xlabel("", fontsize=24)
ax["A"].set_ylabel(r"Efficiency difference ($\Delta \eta_{CM}$)", fontsize=24)
ax["A"].tick_params(axis="y", which="major", labelsize=14)

# Violinplot of efficiency difference for CB null model
sns.violinplot(
    x="network_type",
    y="efficiency_diff_CB",
    data=df_plot_efficiency_bipartite,
    palette=[
        color_dict.get(n_type, "gray") for n_type in df_plot_efficiency_bipartite["network_type"].unique()
    ],
    ax=ax["B"],
)
ax["B"].axhline(0, color="k", ls="--", lw=2)

ax["B"].set_xticklabels(
    [net_type_initials[n_type] for n_type in df_plot_efficiency_bipartite["network_type"].unique()],
)

ax["B"].set_xlabel("", fontsize=24)
ax["B"].set_ylabel(r"Efficiency difference ($\Delta \eta_{CB}$)", fontsize=24)
ax["B"].tick_params(axis="y", which="major", labelsize=14)

## Expected exponent

In [ ]:
expected_exponents = 1 + np.log(3 / (df_plot_efficiency["diffusion_time"] * c.val)) / np.log(
    df_plot_efficiency["S"]
)

expected_exponent = np.mean(expected_exponents)

fig, ax = plt.subplots(figsize=(8, 6))

# Histogram with kde (lw=5) of the expected exponent
sns.histplot(expected_exponents, kde=True, ax=ax, edgecolor="black", bins=20, line_kws={"lw": 4}, stat="density")

# kde line with lw=5

plt.axvline(
    expected_exponent,
    color="k",
    ls="--",
    lw=3,
    label=r"$\langle\gamma_{{opt}}\rangle={:.2f}$".format(expected_exponent),
)

# # Median
# plt.axvline(
#     np.median(expected_exponents),
#     color="red",
#     ls="--",
#     lw=3,
#     label=r"$\gamma_{{median}}={:.2f}$".format(np.median(expected_exponents)),
# )

# # Mode
# plt.axvline(
#     1.3,
#     color="blue",
#     ls="--",
#     lw=3,
#     label=r"$\gamma_{{mode}}={:.2f}$".format(1.3),
# )

plt.xlabel(r"$\gamma_{opt}$", fontsize=24)
plt.ylabel("Density", fontsize=24)

plt.tick_params(axis="both", which="major", labelsize=14)

plt.legend(fontsize=14)

## Map of networks used

In [ ]:
type_to_label = {
    "food_web": "Food Webs",
    "anemone_fish": "Anemone-Fish",
    "plant_pollinator": "Pollination",
    "host_parasite": "Host-Parasite",
    "seed_dispersal": "Seed Dispersal",
    "plant_ant": "Plant-Ant",
}

df_metadata_mangal = pd.read_csv("Data/mangal/mangal_networks_metadata.csv")

df_metadata_mangal["dataset"] = "Mangal"
df_metadata_mangal["id"] = "mangal_" + df_metadata_mangal["id"].astype(str)

df_metadata_wol = pd.DataFrame()

for folder in os.listdir("Data/web_of_life/"):

    df_i = pd.read_csv("Data/web_of_life/{}/references.csv".format(folder))
    
    df_i["dataset"] = "Web of Life"
        
    df_metadata_wol = pd.concat([df_metadata_wol, df_i], axis=0)

df_metadata_mangal.rename(columns={"id": "ID"}, inplace=True)
df_metadata_mangal.rename(columns={"name": "Reference"}, inplace=True)
df_metadata_mangal.rename(columns={"lon": "Longitude"}, inplace=True)
df_metadata_mangal.rename(columns={"lat": "Latitude"}, inplace=True)

# Remove dataset, public, complete, user and description columns from df_metadata_mangal
df_metadata_mangal.drop(columns=["public", "complete", "user", "description", "date"], inplace=True)

# Translate interaction_type to label
df_metadata_mangal["Type of interactions"] = df_metadata_mangal["interaction_type"].map(
    type_to_label
)

# Remove interaction_type column from df_metadata_mangal
df_metadata_mangal.drop(columns=["interaction_type"], inplace=True)

# Remove Species, Interactions, Connectance, Type of data, Locality of Study in df_metadata_wol
df_metadata_wol.drop(columns=["Species", "Interactions", "Connectance", "Type of data", "Locality of Study"], inplace=True)

# combine both metadata dataframes
df_metadata = pd.concat([df_metadata_mangal, df_metadata_wol], axis=0)

# Get number of species from df_network_metrics
df_network_metrics = pd.read_csv("Results/network_metrics.csv")
df_metadata_f = df_metadata.merge(df_network_metrics[["id", "S"]], left_on="ID", right_on="id", how="inner")

df_metadata_f.dropna(inplace=True)

# interaction_type column for plotting
for n_type in df_plot["network_type"].unique():
    df_metadata_f.loc[df_metadata_f["Type of interactions"] == type_to_label[n_type], "network_type"] = n_type

df_metadata_f

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# Map with the location of the networks. Marker depends on the dataset, color depends on the type of interaction, marker size depends on the number of species
dataset_markers = {
    "Mangal": "o",
    "Web of Life": "^",
}

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.Robinson()})

ax.stock_img()

ax.coastlines(resolution="110m")  # 110m, 50m, 10m

ax.add_feature(
    cfeature.OCEAN, edgecolor=None, facecolor="azure", zorder=1
)

ax.set_global()

for dataset in df_metadata_f["dataset"].unique():

    df_subset = df_metadata_f[df_metadata_f["dataset"] == dataset]

    for interaction_type in df_subset["network_type"].unique():

        df_subset_type = df_subset[df_subset["network_type"] == interaction_type]

        ax.scatter(
            df_subset_type["Longitude"],
            df_subset_type["Latitude"],
            s=50,
            color=color_dict.get(interaction_type, "gray"),
            marker=dataset_markers.get(dataset, "o"),
            label="{} - {}".format(dataset, interaction_type),
            zorder=2,
            edgecolors="black",
            transform=ccrs.PlateCarree(),
        )

# Custom legend
handles = []
labels = []

# Use color from interaction_type and label from type_of_interactions
for n_type in df_metadata_f["network_type"].unique():
    handles.append(
        plt.Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=color_dict.get(n_type, "gray"), markersize=10
        )
    )
    labels.append(type_to_label.get(n_type, n_type))

# Add black markers for dataset
for dataset in df_metadata_f["dataset"].unique():
    handles.append(
        plt.Line2D(
            [0], [0], marker=dataset_markers.get(dataset, "o"), color="w", markerfacecolor="black", markersize=10
        )
    )
    labels.append(dataset)

ax.legend(handles, labels, ncol=4, bbox_to_anchor=(0.98, 1.25), fontsize=14)

In [ ]:
# Map with the location of the networks. Marker depends on the dataset, color depends on the type of interaction, marker size depends on the number of species
dataset_markers = {
    "Mangal": "o",
    "Web of Life": "^",
}

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.Robinson()})

# ax.stock_img()

# ax.coastlines(resolution="50m")  # 110m, 50m, 10m

# ax.add_feature(cfeature.BORDERS, edgecolor="black", lw=0.5)
# # ax.add_feature(
# #     cfeature.OCEAN, edgecolor=None, facecolor="azure", zorder=1
# # )

ax.set_global()
ax.set_facecolor("white")

# Very quiet base map
ax.add_feature(
    cfeature.LAND.with_scale("110m"),
    facecolor="0.92",
    edgecolor="k",
    zorder=0,
)

# ax.add_feature(
#     cfeature.OCEAN.with_scale("110m"),
#     facecolor="lightblue",
#     alpha=0.1,
#     edgecolor="none",
#     zorder=0,
# )

ax.coastlines(
    resolution="110m",
    linewidth=0.35,
    color="0.70",
    zorder=1,
)

for dataset in df_metadata_f["dataset"].unique():

    df_subset = df_metadata_f[df_metadata_f["dataset"] == dataset]

    for interaction_type in df_subset["network_type"].unique():

        df_subset_type = df_subset[df_subset["network_type"] == interaction_type]

        ax.scatter(
            df_subset_type["Longitude"],
            df_subset_type["Latitude"],
            s=50,
            color=color_dict.get(interaction_type, "gray"),
            marker=dataset_markers.get(dataset, "o"),
            label="{} - {}".format(dataset, interaction_type),
            zorder=3,
            alpha=0.85,
            linewidths=0.5,
            edgecolors="black",
            transform=ccrs.PlateCarree(),
            rasterized=True,
        )

# Custom legend
handles = []
labels = []

# Use color from interaction_type and label from type_of_interactions
for n_type in df_metadata_f["network_type"].unique():
    handles.append(
        plt.Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=color_dict.get(n_type, "gray"), markersize=10
        )
    )
    labels.append(type_to_label.get(n_type, n_type))

# Add black markers for dataset
for dataset in df_metadata_f["dataset"].unique():
    handles.append(
        plt.Line2D(
            [0], [0], marker=dataset_markers.get(dataset, "o"), color="w", markerfacecolor="black", markersize=10
        )
    )
    labels.append(dataset)

ax.legend(handles, labels, ncol=4, bbox_to_anchor=(0.98, 1.25), fontsize=14)